In [6]:
!pip install earthengine-api geemap
import ee
ee.Authenticate()
ee.Initialize(project='juairia-123')
import geemap

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
Map = geemap.Map(center=[22.0, 90.0], zoom=8)

aoi = ee.Geometry.Rectangle([89.8, 21.6, 90.9, 23.2])
# Load Sentinel-2 image collection, filter and clip
sentinel = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2023-01-01', '2023-12-31') \
    .filterBounds(aoi) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median() \
    .clip(aoi)

# Add layer to map
Map.addLayer(sentinel, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'RGB Sentinel')
Map

Map(center=[22.0, 90.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [3]:
Map = geemap.Map(center=[22.0, 90.0], zoom=8)

#aoi = ee.Geometry.Rectangle([89.3, 21.7, 91.0, 22.8])

#aoi = ee.Geometry.Rectangle([89.8, 21.6, 90.9, 23.2])



#aoi = ee.Geometry.Rectangle([90, 22.0, 90.3, 22.4])#bame,niche,dane,lomba

#aoi = ee.Geometry.Rectangle([90, 22, 90.3, 22.4])
aoi = ee.Geometry.Rectangle([89.83, 22.0, 90.28, 22.4])#Payra, Baleshwar, Bishkhali

sentinel = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2023-01-01', '2023-12-31') \
    .filterBounds(aoi) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median() \
    .clip(aoi)

# Add layer to map
Map.addLayer(sentinel, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'RGB Sentinel')
Map

Map(center=[22.0, 90.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
import ee
import pandas as pd
import geemap

# --- Initialize map ---
Map = geemap.Map(center=[22.15, 90.215], zoom=13)

# --- Define original AOI ---
#original_aoi_3 = ee.Geometry.Rectangle([90.07, 22.1, 90.23, 22.14])
original_aoi_3 = ee.Geometry.Rectangle([90.04,22.05,90.20,22.09])# zone9
# BASELINE YEAR FOR EROSION/ACCRETION
# -------------------------------
baseline_year = 1988

def mask_landsat_sr(image):
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    mask = cloud.Or(shadow).Not()
    return image.updateMask(mask)

def load_landsat5(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    return (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
            .filterDate(start, end)
            .filterBounds(original_aoi_3)
            .map(mask_landsat_sr)
            .map(lambda img: img.multiply(0.0000275).add(-0.2))
            .median()
            .clip(original_aoi_3))

# --- Baseline image and water ---
baseline_img = load_landsat5(baseline_year)
baseline_ndwi = baseline_img.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDWI')
baseline_water_mask = baseline_ndwi.gt(0)
baseline_shifted_mask = baseline_water_mask.focal_max(radius=2, units='pixels')
baseline_water_vector = baseline_shifted_mask.selfMask().reduceToVectors(
    geometry=original_aoi_3, scale=30, geometryType='polygon',
    labelProperty='water', bestEffort=True, maxPixels=1e13
)
baseline_water_vector = baseline_water_vector.map(lambda f: f.set('area', f.geometry().area(1)))
baseline_largest_polygon = baseline_water_vector.sort('area', False).first()
baseline_water_geom = baseline_largest_polygon.geometry()
# --- Load Sentinel-2 collection ---
def load_s2(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    return (ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')
            .filterDate(start, end)
            .filterBounds(original_aoi_3)
            .map(mask_landsat_sr)
            .map(lambda img: img.multiply(0.0000275).add(-0.2))
            .median()
            .clip(original_aoi_3))

# Elevation, slope, soil
elevation = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(elevation)
soil = ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02')
soil_texture_dict = ee.Dictionary({
    1:'Sand',2:'Loamy Sand',3:'Sandy Loam',4:'Loam',5:'Silt Loam',
    6:'Silt',7:'Sandy Clay Loam',8:'Clay Loam',9:'Silty Clay Loam',
    10:'Sandy Clay',11:'Silty Clay',12:'Clay'
})

# Number of trial lines for best line detection
n_points = 30

# Store all stats
all_stats = []

# -------------------------------
# YEAR LOOP
# -------------------------------
for year in [2002, 2012, 2013, 2014, 2015, 2016, 2017]:
    print(f"Processing year: {year}")

    # --- Load Landsat-7 ---
    landsat7 = load_s2(year)
    ndwi = landsat7.normalizedDifference(['SR_B2', 'SR_B4']).rename('NDWI')
    ndvi = landsat7.normalizedDifference(['SR_B4','SR_B3']).rename('NDVI')
    ndbi = landsat7.normalizedDifference(['SR_B5','SR_B4']).rename('NDBI')
    savi = landsat7.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
        {'NIR': landsat7.select('SR_B4'), 'RED': landsat7.select('SR_B3')}
    ).rename('SAVI')
    msi = landsat7.select('SR_B5').divide(landsat7.select('SR_B4')).rename('MSI')
    nbr = landsat7.normalizedDifference(['SR_B4','SR_B7']).rename('NBR')
    bai = landsat7.expression(
        '1 / ((0.1 - RED)**2 + (0.06 - NIR)**2)',
        {'RED': landsat7.select('SR_B3'), 'NIR': landsat7.select('SR_B4')}
    ).rename('BAI')


    # --- Masks ---
    veg_mask = ndvi.gt(0.4).rename('veg_mask')
    builtup_mask = ndbi.gt(0).rename('builtup_mask')
    bare_mask = (veg_mask.eq(0).And(builtup_mask.eq(0))).rename('bare_mask')

    # -------------------------------
    # WATER MASK & LARGEST POLYGON
    # -------------------------------
    water_mask = ndwi.gt(0)
    shifted_water_mask = water_mask.focal_max(radius=2, units='pixels')
    water_vector = shifted_water_mask.selfMask().reduceToVectors(
        geometry=original_aoi_3,
        scale=30,
        geometryType='polygon',
        labelProperty='water',
        bestEffort=True,
        maxPixels=1e13
    )
    water_vector = water_vector.map(lambda f: f.set('area', f.geometry().area(1)))
    largest_polygon = water_vector.sort('area', False).first()
    water_geom = largest_polygon.geometry()
    water_vector_largest = ee.FeatureCollection([largest_polygon])

    # -------------------------------
    # BEST LINE DETECTION
    # -------------------------------
    coords = original_aoi_3.bounds().coordinates().getInfo()[0]
    xmin, ymin = coords[0]
    xmax, ymax = coords[2]

    def generate_lines_and_get_max(type):
        trial_lines = []
        original_lines = []
        if type == 'length':
            for i in range(n_points + 1):
                x = xmin + i * (xmax - xmin)/n_points
                line = ee.Geometry.LineString([[x, ymin], [x, ymax]])
                trial_lines.append(line.intersection(water_geom, 1))
                original_lines.append(line)
        elif type == 'diagonal':
            corners = [[xmin, ymin], [xmin, ymax], [xmax, ymin], [xmax, ymax]]
            for corner in corners:
                cx, cy = corner
                for i in range(n_points + 1):
                    for j in range(n_points + 1):
                        if corner == [xmin, ymin]:
                            ex = xmin + i*(xmax-xmin)/n_points
                            ey = ymin + j*(ymax-ymin)/n_points
                        elif corner == [xmin, ymax]:
                            ex = xmin + i*(xmax-xmin)/n_points
                            ey = ymax - j*(ymax-ymin)/n_points
                        elif corner == [xmax, ymin]:
                            ex = xmax - i*(xmax-xmin)/n_points
                            ey = ymin + j*(ymax-ymin)/n_points
                        else:
                            ex = xmax - i*(xmax-xmin)/n_points
                            ey = ymax - j*(ymax-ymin)/n_points
                        line = ee.Geometry.LineString([[cx, cy], [ex, ey]])
                        trial_lines.append(line.intersection(water_geom, 1))
                        original_lines.append(line)
        max_len, best_clipped, best_original = 0, None, None
        for clipped, original in zip(trial_lines, original_lines):
            length_val = clipped.length().getInfo()
            if length_val > max_len:
                max_len = length_val
                best_clipped = clipped
                best_original = original
        return best_clipped, best_original, max_len

    length_clipped, length_original, length_val = generate_lines_and_get_max('length')
    diagonal_clipped, diagonal_original, diagonal_val = generate_lines_and_get_max('diagonal')

    best_line_clipped = length_clipped
    best_line_original = length_original
    best_type = 'length'
    max_val = length_val
    if diagonal_val > max_val:
        best_line_clipped = diagonal_clipped
        best_line_original = diagonal_original
        best_type = 'diagonal'
        max_val = diagonal_val

    # --- Extract line coordinates ---
    line_coords = best_line_clipped.coordinates().getInfo()
    flat_coords = []
    if isinstance(line_coords[0][0], list):
        for seg in line_coords:
            flat_coords.extend(seg)
    else:
        flat_coords = line_coords
    lons = [pt[0] for pt in flat_coords]
    lats = [pt[1] for pt in flat_coords]

    # --- AOIs ---
    new_xmin, new_xmax = min(lons), max(lons)
    new_ymin, new_ymax = min(lats), max(lats)
    new_aoi_3 = ee.Geometry.Rectangle([new_xmin, new_ymin, new_xmax, new_ymax])
    if best_type == 'length':
        another_aoi_3 = ee.Geometry.Rectangle([xmin, new_ymin, xmax, new_ymax])
    else:
        (x1, y1), (x2, y2) = flat_coords[0], flat_coords[-1]
        if (x2 > x1 and y2 > y1) or (x1 > x2 and y1 > y2):
            another_aoi_3 = ee.Geometry.Rectangle([new_xmin, ymin, new_xmax, ymax])
        else:
            another_aoi_3 = ee.Geometry.Rectangle([xmin, new_ymin, xmax, new_ymax])

    # --- Continuous purple line ---
    fc_original = ee.FeatureCollection([ee.Feature(best_line_original)])
    purple_line_continuous = fc_original.union().geometry().simplify(1)

    # -------------------------------
    # BANK CONTINUOUS & CLIPPING
    # -------------------------------
    bank_continuous = (water_geom.buffer(200).difference(water_geom, 1)
                       .intersection(another_aoi_3, 1))

    # -------------------------------
    # SPLIT AOI USING PURPLE LINE
    # -------------------------------
    line_buffer = purple_line_continuous.buffer(0.00005)
    split_aois = another_aoi_3.difference(line_buffer, 1)
    portions = ee.List(split_aois.geometries())

    def label_portions(portions):
        coords = purple_line_continuous.coordinates()
        start = ee.List(coords.get(0))
        end = ee.List(coords.get(-1))
        x0 = ee.Number(start.get(0))
        y0 = ee.Number(start.get(1))
        x1 = ee.Number(end.get(0))
        y1 = ee.Number(end.get(1))
        # Determine diagonal type
        diag_type = ee.Algorithms.If((x1.subtract(x0)).multiply(y1.subtract(y0)).lt(0), "\\", "/")
        lat0 = ee.Number(ee.Geometry(portions.get(0)).centroid(1).coordinates().get(1))
        lat1 = ee.Number(ee.Geometry(portions.get(1)).centroid(1).coordinates().get(1))
        upper_index = ee.Algorithms.If(lat0.gt(lat1), 0, 1)
        lower_index = ee.Algorithms.If(lat0.gt(lat1), 1, 0)
        return ee.Algorithms.If(
            ee.String(diag_type).compareTo("\\").eq(0),
            ee.List([lower_index, upper_index]),
            ee.List([upper_index, lower_index])
        )

    order = ee.List(label_portions(portions))
    left_geom = ee.Geometry(portions.get(order.get(0)))
    right_geom = ee.Geometry(portions.get(order.get(1)))

    # -------------------------------
    # EROSION / ACCRETION PER BANK
    # -------------------------------

    # Left bank retreat/accretion
    left_retreat_geom = baseline_water_geom.intersection(left_geom, 1).difference(
        water_geom.intersection(left_geom, 1), 1
    )
    left_accretion_geom = water_geom.intersection(left_geom, 1).difference(
        baseline_water_geom.intersection(left_geom, 1), 1
    )

    left_ret_area = left_retreat_geom.area(1)
    left_acc_area = left_accretion_geom.area(1)
    #left_total_area = baseline_water_geom.intersection(left_geom, 1).area(1)

    #left_ret_pct = ee.Number(left_ret_area).divide(left_total_area).multiply(100)
    #left_acc_pct = ee.Number(left_acc_area).divide(left_total_area).multiply(100)

    # Right bank retreat/accretion
    right_retreat_geom = baseline_water_geom.intersection(right_geom, 1).difference(
        water_geom.intersection(right_geom, 1), 1
    )
    right_accretion_geom = water_geom.intersection(right_geom, 1).difference(
        baseline_water_geom.intersection(right_geom, 1), 1
    )

    right_ret_area = right_retreat_geom.area(1)
    right_acc_area = right_accretion_geom.area(1)
    #right_total_area = baseline_water_geom.intersection(right_geom, 1).area(1)

    #right_ret_pct = ee.Number(right_ret_area).divide(right_total_area).multiply(100)
    #right_acc_pct = ee.Number(right_acc_area).divide(right_total_area).multiply(100)
    # -------------------------------
    # CLASSIFY BANKS
    # -------------------------------
    def assign_side_using_portions(bank_geom, left_portion, right_portion):
        geom_obj = ee.Geometry(bank_geom)
        centroid = geom_obj.centroid(1)
        is_in_left = left_portion.contains(centroid)
        is_in_right = right_portion.contains(centroid)
        side = ee.Algorithms.If(is_in_left, 'left',
               ee.Algorithms.If(is_in_right, 'right', 'unknown'))
        return ee.Feature(geom_obj).set('side', side)

    classified_banks = ee.FeatureCollection(
        bank_continuous.geometries().map(
            lambda g: assign_side_using_portions(g, left_geom, right_geom)
        )
    )
    left_banks = classified_banks.filter(ee.Filter.eq('side', 'left'))
    right_banks = classified_banks.filter(ee.Filter.eq('side', 'right'))

    # -------------------------------
    # BANK STATS FUNCTION
    # -------------------------------
    def compute_bank_stats(bank_geom):
        total_area = bank_geom.area(1)
        veg_area = veg_mask.multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=bank_geom, scale=30, maxPixels=1e13
        ).get('veg_mask')
        builtup_area = builtup_mask.multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=bank_geom, scale=30, maxPixels=1e13
        ).get('builtup_mask')
        bare_area = bare_mask.multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=bank_geom, scale=30, maxPixels=1e13
        ).get('bare_mask')
        percent_veg = ee.Number(veg_area).divide(total_area).multiply(100)
        percent_builtup = ee.Number(builtup_area).divide(total_area).multiply(100)
        percent_bare = ee.Number(bare_area).divide(total_area).multiply(100)
        mean_ndvi = ndvi.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('NDVI')
        mean_ndbi = ndbi.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('NDBI')
        mean_savi = savi.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('SAVI')
        mean_msi = msi.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('MSI')
        mean_nbr = nbr.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('NBR')
        mean_bai = bai.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('BAI')
        mean_elev = elevation.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('elevation')
        mean_slope = slope.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('slope')
        soil_hist = ee.Dictionary(soil.reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(),
            geometry=bank_geom,
            scale=30,
            maxPixels=1e13
        ).get('b0'))
        soil_keys = soil_hist.keys()
        soil_values = soil_hist.values()
        max_value = soil_values.reduce(ee.Reducer.max())
        dominant_index = soil_values.indexOf(max_value)
        soil_code = ee.Number(soil_keys.get(dominant_index))
        soil_class = soil_texture_dict.get(soil_code)
        return {
            'area_m2': total_area,
            'veg_m2': veg_area,
            'builtup_m2': builtup_area,
            'bare_m2': bare_area,
            'percent_veg': percent_veg,
            'percent_builtup': percent_builtup,
            'percent_bare': percent_bare,
            'mean_ndvi': mean_ndvi,
            'mean_ndbi': mean_ndbi,
            'mean_savi': mean_savi,
            'mean_msi': mean_msi,
            'mean_nbr': mean_nbr,
            'mean_bai': mean_bai,
            'mean_elevation': mean_elev,
            'mean_slope': mean_slope,
            'soil_code': soil_code,
            'soil_class': soil_class
        }

    # -------------------------------
    # COMPUTE LEFT & RIGHT BANKS
    # -------------------------------
    left_stats = compute_bank_stats(left_geom)
    right_stats = compute_bank_stats(right_geom)

    for bank_name, stats in zip(['Left', 'Right'], [left_stats, right_stats]):
        row = {k:(v.getInfo() if isinstance(v, ee.ComputedObject) else v) for k,v in stats.items()}
        row['bank'] = bank_name
        row['river'] = 'Payra_Zone9' #also add zone here
        row['year'] = year
        if bank_name == 'Left':
            row['retreat(erosion)_area_m2'] = left_ret_area.getInfo()
            row['accretion_area_m2'] = left_acc_area.getInfo()

        else:
            row['retreat(erosion)_area_m2'] = right_ret_area.getInfo()
            row['accretion_area_m2'] = right_acc_area.getInfo()
        all_stats.append(row)

# -------------------------------
# SAVE CSV
# -------------------------------
df = pd.DataFrame(all_stats)
print("\n=== LEFT & RIGHT BANK LAND COVER & INDICES ===")
print(df.to_string(index=False))
df.to_csv("Payra_Zone9_Landsat_7_2002_2012_2017.csv", index=False)
print("Saved results to CSV")
# Save to Google Drive
drive_path = "/content/drive/MyDrive/Payra_Zone9_Landsat_7_2002_2012_2017.csv"
df.to_csv(drive_path, index=False)
print(f"Saved results to Google Drive at {drive_path}")

In [ ]:
import ee
import pandas as pd
import geemap

# --- Initialize map ---
Map = geemap.Map(center=[22.15, 90.215], zoom=13)

# --- Define original AOI ---
original_aoi_3 = ee.Geometry.Rectangle([90.07, 22.1, 90.23, 22.14])
# BASELINE YEAR FOR EROSION/ACCRETION
# -------------------------------
baseline_year = 1988

def mask_landsat_sr(image):
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    mask = cloud.Or(shadow).Not()
    return image.updateMask(mask)

def load_landsat5(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    return (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
            .filterDate(start, end)
            .filterBounds(original_aoi_3)
            .map(mask_landsat_sr)
            .map(lambda img: img.multiply(0.0000275).add(-0.2))
            .median()
            .clip(original_aoi_3))

# --- Baseline image and water ---
baseline_img = load_landsat5(baseline_year)
baseline_ndwi = baseline_img.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDWI')
baseline_water_mask = baseline_ndwi.gt(0)
baseline_shifted_mask = baseline_water_mask.focal_max(radius=2, units='pixels')
baseline_water_vector = baseline_shifted_mask.selfMask().reduceToVectors(
    geometry=original_aoi_3, scale=30, geometryType='polygon',
    labelProperty='water', bestEffort=True, maxPixels=1e13
)
baseline_water_vector = baseline_water_vector.map(lambda f: f.set('area', f.geometry().area(1)))
baseline_largest_polygon = baseline_water_vector.sort('area', False).first()
baseline_water_geom = baseline_largest_polygon.geometry()
# --- Load Sentinel-2 collection ---
def load_s2(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    return (ee.ImageCollection('COPERNICUS/S2_SR')
            .filterDate(start, end)
            .filterBounds(original_aoi_3)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
            .map(lambda img: img.select(['B3','B4','B8','B11','B12']))
            .median()
            .clip(original_aoi_3))

# Elevation, slope, soil
elevation = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(elevation)
soil = ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02')
soil_texture_dict = ee.Dictionary({
    1:'Sand',2:'Loamy Sand',3:'Sandy Loam',4:'Loam',5:'Silt Loam',
    6:'Silt',7:'Sandy Clay Loam',8:'Clay Loam',9:'Silty Clay Loam',
    10:'Sandy Clay',11:'Silty Clay',12:'Clay'
})

# Number of trial lines for best line detection
n_points = 30

# Store all stats
all_stats = []

# -------------------------------
# YEAR LOOP
# -------------------------------
for year in range(2018, 2026):
    print(f"Processing year: {year}")

    # --- Load Sentinel-2 ---
    sentinel2 = load_s2(year)

    # --- Indices ---
    ndwi = sentinel2.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndvi = sentinel2.normalizedDifference(['B8','B4']).rename('NDVI')
    ndbi = sentinel2.normalizedDifference(['B11','B8']).rename('NDBI')
    savi = sentinel2.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
        {'NIR': sentinel2.select('B8'), 'RED': sentinel2.select('B4')}
    ).rename('SAVI')
    msi = sentinel2.select('B11').divide(sentinel2.select('B8')).rename('MSI')
    nbr = sentinel2.normalizedDifference(['B8','B12']).rename('NBR')
    bai = sentinel2.expression(
        '1 / ((0.1 - RED)**2 + (0.06 - NIR)**2)',
        {'RED': sentinel2.select('B4'), 'NIR': sentinel2.select('B8')}
    ).rename('BAI')

    # --- Masks ---
    veg_mask = ndvi.gt(0.4).rename('veg_mask')
    builtup_mask = ndbi.gt(0).rename('builtup_mask')
    bare_mask = (veg_mask.eq(0).And(builtup_mask.eq(0))).rename('bare_mask')

    # -------------------------------
    # WATER MASK & LARGEST POLYGON
    # -------------------------------
    water_mask = ndwi.gt(0)
    shifted_water_mask = water_mask.focal_max(radius=2, units='pixels')
    water_vector = shifted_water_mask.selfMask().reduceToVectors(
        geometry=original_aoi_3,
        scale=30,
        geometryType='polygon',
        labelProperty='water',
        bestEffort=True,
        maxPixels=1e13
    )
    water_vector = water_vector.map(lambda f: f.set('area', f.geometry().area(1)))
    largest_polygon = water_vector.sort('area', False).first()
    water_geom = largest_polygon.geometry()
    water_vector_largest = ee.FeatureCollection([largest_polygon])

    # -------------------------------
    # BEST LINE DETECTION
    # -------------------------------
    coords = original_aoi_3.bounds().coordinates().getInfo()[0]
    xmin, ymin = coords[0]
    xmax, ymax = coords[2]

    def generate_lines_and_get_max(type):
        trial_lines = []
        original_lines = []
        if type == 'length':
            for i in range(n_points + 1):
                x = xmin + i * (xmax - xmin)/n_points
                line = ee.Geometry.LineString([[x, ymin], [x, ymax]])
                trial_lines.append(line.intersection(water_geom, 1))
                original_lines.append(line)
        elif type == 'diagonal':
            corners = [[xmin, ymin], [xmin, ymax], [xmax, ymin], [xmax, ymax]]
            for corner in corners:
                cx, cy = corner
                for i in range(n_points + 1):
                    for j in range(n_points + 1):
                        if corner == [xmin, ymin]:
                            ex = xmin + i*(xmax-xmin)/n_points
                            ey = ymin + j*(ymax-ymin)/n_points
                        elif corner == [xmin, ymax]:
                            ex = xmin + i*(xmax-xmin)/n_points
                            ey = ymax - j*(ymax-ymin)/n_points
                        elif corner == [xmax, ymin]:
                            ex = xmax - i*(xmax-xmin)/n_points
                            ey = ymin + j*(ymax-ymin)/n_points
                        else:
                            ex = xmax - i*(xmax-xmin)/n_points
                            ey = ymax - j*(ymax-ymin)/n_points
                        line = ee.Geometry.LineString([[cx, cy], [ex, ey]])
                        trial_lines.append(line.intersection(water_geom, 1))
                        original_lines.append(line)
        max_len, best_clipped, best_original = 0, None, None
        for clipped, original in zip(trial_lines, original_lines):
            length_val = clipped.length().getInfo()
            if length_val > max_len:
                max_len = length_val
                best_clipped = clipped
                best_original = original
        return best_clipped, best_original, max_len

    length_clipped, length_original, length_val = generate_lines_and_get_max('length')
    diagonal_clipped, diagonal_original, diagonal_val = generate_lines_and_get_max('diagonal')

    best_line_clipped = length_clipped
    best_line_original = length_original
    best_type = 'length'
    max_val = length_val
    if diagonal_val > max_val:
        best_line_clipped = diagonal_clipped
        best_line_original = diagonal_original
        best_type = 'diagonal'
        max_val = diagonal_val

    # --- Extract line coordinates ---
    line_coords = best_line_clipped.coordinates().getInfo()
    flat_coords = []
    if isinstance(line_coords[0][0], list):
        for seg in line_coords:
            flat_coords.extend(seg)
    else:
        flat_coords = line_coords
    lons = [pt[0] for pt in flat_coords]
    lats = [pt[1] for pt in flat_coords]

    # --- AOIs ---
    new_xmin, new_xmax = min(lons), max(lons)
    new_ymin, new_ymax = min(lats), max(lats)
    new_aoi_3 = ee.Geometry.Rectangle([new_xmin, new_ymin, new_xmax, new_ymax])
    if best_type == 'length':
        another_aoi_3 = ee.Geometry.Rectangle([xmin, new_ymin, xmax, new_ymax])
    else:
        (x1, y1), (x2, y2) = flat_coords[0], flat_coords[-1]
        if (x2 > x1 and y2 > y1) or (x1 > x2 and y1 > y2):
            another_aoi_3 = ee.Geometry.Rectangle([new_xmin, ymin, new_xmax, ymax])
        else:
            another_aoi_3 = ee.Geometry.Rectangle([xmin, new_ymin, xmax, new_ymax])

    # --- Continuous purple line ---
    fc_original = ee.FeatureCollection([ee.Feature(best_line_original)])
    purple_line_continuous = fc_original.union().geometry().simplify(1)

    # -------------------------------
    # BANK CONTINUOUS & CLIPPING
    # -------------------------------
    bank_continuous = (water_geom.buffer(200).difference(water_geom, 1)
                       .intersection(another_aoi_3, 1))

    # -------------------------------
    # SPLIT AOI USING PURPLE LINE
    # -------------------------------
    line_buffer = purple_line_continuous.buffer(0.00005)
    split_aois = another_aoi_3.difference(line_buffer, 1)
    portions = ee.List(split_aois.geometries())

    def label_portions(portions):
        coords = purple_line_continuous.coordinates()
        start = ee.List(coords.get(0))
        end = ee.List(coords.get(-1))
        x0 = ee.Number(start.get(0))
        y0 = ee.Number(start.get(1))
        x1 = ee.Number(end.get(0))
        y1 = ee.Number(end.get(1))
        # Determine diagonal type
        diag_type = ee.Algorithms.If((x1.subtract(x0)).multiply(y1.subtract(y0)).lt(0), "\\", "/")
        lat0 = ee.Number(ee.Geometry(portions.get(0)).centroid(1).coordinates().get(1))
        lat1 = ee.Number(ee.Geometry(portions.get(1)).centroid(1).coordinates().get(1))
        upper_index = ee.Algorithms.If(lat0.gt(lat1), 0, 1)
        lower_index = ee.Algorithms.If(lat0.gt(lat1), 1, 0)
        return ee.Algorithms.If(
            ee.String(diag_type).compareTo("\\").eq(0),
            ee.List([lower_index, upper_index]),
            ee.List([upper_index, lower_index])
        )

    order = ee.List(label_portions(portions))
    left_geom = ee.Geometry(portions.get(order.get(0)))
    right_geom = ee.Geometry(portions.get(order.get(1)))

    # -------------------------------
    # EROSION / ACCRETION PER BANK
    # -------------------------------

    # Left bank retreat/accretion
    left_retreat_geom = baseline_water_geom.intersection(left_geom, 1).difference(
        water_geom.intersection(left_geom, 1), 1
    )
    left_accretion_geom = water_geom.intersection(left_geom, 1).difference(
        baseline_water_geom.intersection(left_geom, 1), 1
    )

    left_ret_area = left_retreat_geom.area(1)
    left_acc_area = left_accretion_geom.area(1)
    #left_total_area = baseline_water_geom.intersection(left_geom, 1).area(1)

    #left_ret_pct = ee.Number(left_ret_area).divide(left_total_area).multiply(100)
    #left_acc_pct = ee.Number(left_acc_area).divide(left_total_area).multiply(100)

    # Right bank retreat/accretion
    right_retreat_geom = baseline_water_geom.intersection(right_geom, 1).difference(
        water_geom.intersection(right_geom, 1), 1
    )
    right_accretion_geom = water_geom.intersection(right_geom, 1).difference(
        baseline_water_geom.intersection(right_geom, 1), 1
    )

    right_ret_area = right_retreat_geom.area(1)
    right_acc_area = right_accretion_geom.area(1)
    #right_total_area = baseline_water_geom.intersection(right_geom, 1).area(1)

    #right_ret_pct = ee.Number(right_ret_area).divide(right_total_area).multiply(100)
    #right_acc_pct = ee.Number(right_acc_area).divide(right_total_area).multiply(100)
    # -------------------------------
    # CLASSIFY BANKS
    # -------------------------------
    def assign_side_using_portions(bank_geom, left_portion, right_portion):
        geom_obj = ee.Geometry(bank_geom)
        centroid = geom_obj.centroid(1)
        is_in_left = left_portion.contains(centroid)
        is_in_right = right_portion.contains(centroid)
        side = ee.Algorithms.If(is_in_left, 'left',
               ee.Algorithms.If(is_in_right, 'right', 'unknown'))
        return ee.Feature(geom_obj).set('side', side)

    classified_banks = ee.FeatureCollection(
        bank_continuous.geometries().map(
            lambda g: assign_side_using_portions(g, left_geom, right_geom)
        )
    )
    left_banks = classified_banks.filter(ee.Filter.eq('side', 'left'))
    right_banks = classified_banks.filter(ee.Filter.eq('side', 'right'))

    # -------------------------------
    # BANK STATS FUNCTION
    # -------------------------------
    def compute_bank_stats(bank_geom):
        total_area = bank_geom.area(1)
        veg_area = veg_mask.multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=bank_geom, scale=30, maxPixels=1e13
        ).get('veg_mask')
        builtup_area = builtup_mask.multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=bank_geom, scale=30, maxPixels=1e13
        ).get('builtup_mask')
        bare_area = bare_mask.multiply(ee.Image.pixelArea()).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=bank_geom, scale=30, maxPixels=1e13
        ).get('bare_mask')
        percent_veg = ee.Number(veg_area).divide(total_area).multiply(100)
        percent_builtup = ee.Number(builtup_area).divide(total_area).multiply(100)
        percent_bare = ee.Number(bare_area).divide(total_area).multiply(100)
        mean_ndvi = ndvi.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('NDVI')
        mean_ndbi = ndbi.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('NDBI')
        mean_savi = savi.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('SAVI')
        mean_msi = msi.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('MSI')
        mean_nbr = nbr.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('NBR')
        mean_bai = bai.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('BAI')
        mean_elev = elevation.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('elevation')
        mean_slope = slope.reduceRegion(ee.Reducer.mean(), bank_geom, 30, maxPixels=1e13).get('slope')
        soil_hist = ee.Dictionary(soil.reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(),
            geometry=bank_geom,
            scale=30,
            maxPixels=1e13
        ).get('b0'))
        soil_keys = soil_hist.keys()
        soil_values = soil_hist.values()
        max_value = soil_values.reduce(ee.Reducer.max())
        dominant_index = soil_values.indexOf(max_value)
        soil_code = ee.Number(soil_keys.get(dominant_index))
        soil_class = soil_texture_dict.get(soil_code)
        return {
            'area_m2': total_area,
            'veg_m2': veg_area,
            'builtup_m2': builtup_area,
            'bare_m2': bare_area,
            'percent_veg': percent_veg,
            'percent_builtup': percent_builtup,
            'percent_bare': percent_bare,
            'mean_ndvi': mean_ndvi,
            'mean_ndbi': mean_ndbi,
            'mean_savi': mean_savi,
            'mean_msi': mean_msi,
            'mean_nbr': mean_nbr,
            'mean_bai': mean_bai,
            'mean_elevation': mean_elev,
            'mean_slope': mean_slope,
            'soil_code': soil_code,
            'soil_class': soil_class
        }

    # -------------------------------
    # COMPUTE LEFT & RIGHT BANKS
    # -------------------------------
    left_stats = compute_bank_stats(left_geom)
    right_stats = compute_bank_stats(right_geom)

    for bank_name, stats in zip(['Left', 'Right'], [left_stats, right_stats]):
        row = {k:(v.getInfo() if isinstance(v, ee.ComputedObject) else v) for k,v in stats.items()}
        row['bank'] = bank_name
        row['river'] = 'Payra'
        row['year'] = year
        if bank_name == 'Left':
            row['retreat(erosion)_area_m2'] = left_ret_area.getInfo()
            row['accretion_area_m2'] = left_acc_area.getInfo()

        else:
            row['retreat(erosion)_area_m2'] = right_ret_area.getInfo()
            row['accretion_area_m2'] = right_acc_area.getInfo()
        all_stats.append(row)

# -------------------------------
# SAVE CSV
# -------------------------------
df = pd.DataFrame(all_stats)
print("\n=== LEFT & RIGHT BANK LAND COVER & INDICES ===")
print(df.to_string(index=False))
df.to_csv("Payra_Zone9_sentinal2_18_25.csv", index=False)
print("Saved results to CSV")
# Save to Google Drive
drive_path = "/content/drive/MyDrive/Payra_Zone9_sentinal2_18_25.csv"
df.to_csv(drive_path, index=False)
print(f"Saved results to Google Drive at {drive_path}")

In [ ]:
import pandas as pd

# Load dataset
data = pd.read_csv("final_merged_all_data.csv")

# --- Step 1: Normalize capitalization and spacing ---
data['bank'] = data['bank'].str.lower().str.strip()
data['river'] = data['river'].str.strip()
data['zone'] = data['zone'].astype(str).str.strip()

# --- Step 2: Detect duplicate year rows before removal ---
duplicate_rows = data[data.duplicated(subset=['river', 'zone', 'bank', 'year'], keep=False)]
removed_duplicates = duplicate_rows.drop_duplicates(subset=['river', 'zone', 'bank'])

# --- Step 3: Remove duplicates (keep first) ---
data = data.drop_duplicates(subset=['river', 'zone', 'bank', 'year'], keep='first')

# --- Step 4: Count entries per group ---
count_entries = data.groupby(['river', 'zone', 'bank']).size().reset_index(name='count')

# --- Step 5: Identify incomplete (<38) groups ---
incomplete_groups = count_entries[count_entries['count'] < 38]

# --- Step 6: Keep only complete (≥38) groups ---
count_entries = count_entries[count_entries['count'] >= 38]

# --- Step 7: Keep only valid data ---
valid_groups = count_entries[['river', 'zone', 'bank']]
filtered_data = pd.merge(data, valid_groups, on=['river', 'zone', 'bank'], how='inner')

# --- Step 8: Sort properly ---
bank_order = {'left': 1, 'right': 2}
filtered_data['bank_order'] = filtered_data['bank'].map(bank_order).fillna(3)
filtered_data['zone'] = filtered_data['zone'].astype(int)
filtered_data = filtered_data.sort_values(by=['river', 'zone', 'bank_order', 'year'])
filtered_data = filtered_data.drop(columns=['bank_order'])

# --- Step 9: Save final cleaned & sorted dataset ---
filtered_data.to_csv("final_merged_all_data_cleaned_sorted.csv", index=False)

# --- Step 10: Report removed items ---
print("\n===== CLEANING SUMMARY =====")

if not removed_duplicates.empty:
    print("\nRemoved duplicate rows (same river, zone, bank, year) for:")
    for _, row in removed_duplicates.iterrows():
        print(f"  - {row['river']} | Zone {row['zone']} | Bank {row['bank']}")

if not incomplete_groups.empty:
    print("\nRemoved incomplete groups (<38 years of data) for:")
    for _, row in incomplete_groups.iterrows():
        print(f"  - {row['river']} | Zone {row['zone']} | Bank {row['bank']} ({row['count']} years)")

if removed_duplicates.empty and incomplete_groups.empty:
    print("✅ No duplicates or incomplete groups removed — all data consistent!")

print("\nCleaned and sorted data saved as 'final_merged_all_data_cleaned_sorted.csv'")